# Timing Log Analysis

This notebook parses stage-level timing lines from legacy and optimized training logs.

Expected timing line format:

```text
[timing] stage=<stage> epoch=<i>/<n> train_sec=<seconds> val_sec=<seconds> total_sec=<seconds>
```

Notes:
- The benchmark order is predefined by `scripts/run_speed_benchmark.sh`.
- `train_sec` and `val_sec` are used for the analysis.
- `total_sec` is ignored because it mixes stage bookkeeping with train/validation work and is not useful for the comparison.


In [ ]:
from __future__ import annotations

import csv
import re
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

ROOT = Path.cwd()
ANALYSIS_DIR = ROOT if ROOT.name == "analysis" else ROOT / "analysis"
LEGACY_LOG = ANALYSIS_DIR / "legacy.log"
OPTIMIZED_LOG = ANALYSIS_DIR / "optimized.log"

TIMING_RE = re.compile(
    r"^\[timing\]\s+stage=(?P<stage>\S+)\s+epoch=(?P<epoch>\d+)/(?P<epochs>\d+)\s+"
    r"train_sec=(?P<train>[\d.]+)\s+val_sec=(?P<val>[\d.]+)\s+"
    r"total_sec=[\d.]+"
)

BENCHMARK_ORDER = [
    f"{dataset}-{model}"
    for dataset in ("synthetic", "CUB", "cifar10")
    for model in ("AR", "CEM", "CBM", "SCBM-amortized", "SCBM-global")
]

@dataclass(frozen=True)
class TimingRow:
    source: str
    experiment: str
    stage: str
    epoch: int
    epochs: int
    train_sec: float
    val_sec: float

LEGACY_LOG, OPTIMIZED_LOG

In [2]:
def parse_timing_file(path: Path, source: str) -> list[TimingRow]:
    rows: list[TimingRow] = []
    experiment_idx = -1
    last_epoch = None

    for line_no, line in enumerate(path.read_text(errors="replace").splitlines(), start=1):
        match = TIMING_RE.match(line)
        if match is None:
            continue

        epoch = int(match.group("epoch"))
        stage = match.group("stage")

        # Each benchmark sub-run starts with epoch 1. AR has three stages, so
        # only advance the experiment for AR pretrain or joint-only runs.
        if epoch == 1 and (
            last_epoch is None
            or stage in ("ar_concept_pretrain", "joint")
        ):
            experiment_idx += 1

        if experiment_idx >= len(BENCHMARK_ORDER):
            raise ValueError(
                f"{path}:{line_no}: more timing blocks than expected benchmark order."
            )

        rows.append(
            TimingRow(
                source=source,
                experiment=BENCHMARK_ORDER[experiment_idx],
                stage=stage,
                epoch=epoch,
                epochs=int(match.group("epochs")),
                train_sec=float(match.group("train")),
                val_sec=float(match.group("val")),
            )
        )
        last_epoch = epoch

    expected_experiments = len(BENCHMARK_ORDER)
    actual_experiments = experiment_idx + 1
    if actual_experiments != expected_experiments:
        raise ValueError(
            f"{path}: parsed {actual_experiments} experiments, expected {expected_experiments}."
        )

    return rows

def summarize(rows: list[TimingRow]) -> dict[tuple[str, str, str], dict[str, float]]:
    acc: dict[tuple[str, str, str], dict[str, float]] = defaultdict(
        lambda: {
            "epochs": 0,
            "train_total": 0.0,
            "val_total": 0.0,
        }
    )
    for row in rows:
        key = (row.source, row.experiment, row.stage)
        acc[key]["epochs"] += 1
        acc[key]["train_total"] += row.train_sec
        acc[key]["val_total"] += row.val_sec

    for values in acc.values():
        n = values["epochs"]
        values["train_per_epoch"] = values["train_total"] / n
        values["val_per_epoch"] = values["val_total"] / n

    return dict(acc)

In [3]:
def write_stage_summary(path: Path, summary: dict[tuple[str, str, str], dict[str, float]]) -> None:
    fieldnames = [
        "source",
        "experiment",
        "stage",
        "epochs",
        "train_total",
        "val_total",
        "train_per_epoch",
        "val_per_epoch",
    ]
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
        writer.writeheader()
        for source, experiment, stage in sorted(summary):
            values = summary[(source, experiment, stage)]
            writer.writerow(
                {
                    "source": source,
                    "experiment": experiment,
                    "stage": stage,
                    "epochs": int(values["epochs"]),
                    **{
                        key: f"{values[key]:.3f}"
                        for key in fieldnames
                        if key not in ("source", "experiment", "stage", "epochs")
                    },
                }
            )

def write_comparison(
    path: Path,
    legacy: dict[tuple[str, str, str], dict[str, float]],
    optimized: dict[tuple[str, str, str], dict[str, float]],
) -> None:
    keys = sorted(
        {
            (experiment, stage)
            for _, experiment, stage in set(legacy) | set(optimized)
        }
    )
    fieldnames = [
        "experiment",
        "stage",
        "legacy_train_total",
        "optimized_train_total",
        "train_speedup",
        "legacy_val_total",
        "optimized_val_total",
        "val_speedup",
    ]
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
        writer.writeheader()
        for experiment, stage in keys:
            legacy_values = legacy.get(("legacy", experiment, stage))
            optimized_values = optimized.get(("optimized", experiment, stage))
            if legacy_values is None or optimized_values is None:
                continue

            def speedup(metric: str) -> str:
                denom = round(optimized_values[metric], 1)
                if denom == 0:
                    return "inf"
                return f"{round(legacy_values[metric], 1) / denom:.1f}"

            writer.writerow(
                {
                    "experiment": experiment,
                    "stage": stage,
                    "legacy_train_total": f"{legacy_values['train_total']:.1f}",
                    "optimized_train_total": f"{optimized_values['train_total']:.1f}",
                    "train_speedup": speedup("train_total"),
                    "legacy_val_total": f"{legacy_values['val_total']:.1f}",
                    "optimized_val_total": f"{optimized_values['val_total']:.1f}",
                    "val_speedup": speedup("val_total"),
                }
            )

In [4]:
def print_markdown_comparison(comparison_path: Path) -> None:
    with comparison_path.open() as f:
        rows = list(csv.DictReader(f, delimiter="\t"))

    print("| experiment | stage | legacy train | optimized train | train speedup |")
    print("| --- | --- | ---: | ---: | ---: |")
    for row in rows:
        print(
            "| {experiment} | {stage} | {legacy_train_total}s | "
            "{optimized_train_total}s | {train_speedup}x |".format(**row)
        )


In [ ]:
legacy_rows = parse_timing_file(LEGACY_LOG, "legacy")
optimized_rows = parse_timing_file(OPTIMIZED_LOG, "optimized")

legacy_summary = summarize(legacy_rows)
optimized_summary = summarize(optimized_rows)
merged_summary = legacy_summary | optimized_summary

summary_out = ANALYSIS_DIR / "timing_stage_summary.tsv"
comparison_out = ANALYSIS_DIR / "timing_comparison.tsv"

write_stage_summary(summary_out, merged_summary)
write_comparison(comparison_out, legacy_summary, optimized_summary)

print_markdown_comparison(comparison_out)
print()
print(f"Wrote {summary_out}")
print(f"Wrote {comparison_out}")


In [6]:
# Inspect raw parsed rows if needed.
import pandas as pd

raw_rows = pd.DataFrame([row.__dict__ for row in legacy_rows + optimized_rows])
raw_rows.head(), raw_rows.groupby(["source", "experiment"]).size().head()

(   source    experiment                stage  epoch  epochs  train_sec  \
 0  legacy  synthetic-AR  ar_concept_pretrain      1       5     10.197   
 1  legacy  synthetic-AR  ar_concept_pretrain      2       5     10.385   
 2  legacy  synthetic-AR  ar_concept_pretrain      3       5      9.882   
 3  legacy  synthetic-AR  ar_concept_pretrain      4       5      9.340   
 4  legacy  synthetic-AR  ar_concept_pretrain      5       5      9.922   
 
    val_sec  
 0   53.091  
 1    0.000  
 2    0.000  
 3    0.000  
 4    0.000  ,
 source  experiment        
 legacy  CUB-AR                15
         CUB-CBM                5
         CUB-CEM                5
         CUB-SCBM-amortized     5
         CUB-SCBM-global        5
 dtype: int64)

In [7]:
# Inspect the stage summary table.
stage_summary = pd.read_csv(summary_out, sep="\t")
stage_summary

,source,experiment,stage,epochs,train_total,val_total,train_per_epoch,val_per_epoch
0,legacy,CUB-AR,ar_concept_pretrain,5,43.829,18.490,8.766,3.698
1,legacy,CUB-AR,concept,5,52.371,17.063,10.474,3.413
2,legacy,CUB-AR,target,5,20.579,17.671,4.116,3.534
3,legacy,CUB-CBM,joint,5,35.987,4.447,7.197,0.889
4,legacy,CUB-CEM,joint,5,67.286,4.295,13.457,0.859
5,legacy,CUB-SCBM-amortized,joint,5,23.686,4.531,4.737,0.906
6,legacy,CUB-SCBM-global,joint,5,20.234,4.837,4.047,0.967
7,legacy,cifar10-AR,ar_concept_pretrain,5,188.242,93.075,37.648,18.615
8,legacy,cifar10-AR,concept,5,198.423,87.321,39.685,17.464
9,legacy,cifar10-AR,target,5,111.792,88.190,22.358,17.638


In [8]:
# Inspect the comparison table.
comparison = pd.read_csv(comparison_out, sep="\t")
comparison

,experiment,stage,legacy_train_total,optimized_train_total,train_speedup,legacy_val_total,optimized_val_total,val_speedup
0,CUB-AR,ar_concept_pretrain,43.8,8.6,5.1,18.5,3.9,4.7
1,CUB-AR,concept,52.4,13.9,3.8,17.1,1.5,11.4
2,CUB-AR,target,20.6,1.3,15.8,17.7,1.5,11.8
3,CUB-CBM,joint,36.0,15.6,2.3,4.4,3.3,1.3
4,CUB-CEM,joint,67.3,15.5,4.3,4.3,3.4,1.3
5,CUB-SCBM-amortized,joint,23.7,17.8,1.3,4.5,3.5,1.3
6,CUB-SCBM-global,joint,20.2,16.4,1.2,4.8,3.8,1.3
7,cifar10-AR,ar_concept_pretrain,188.2,8.7,21.6,93.1,4.1,22.7
8,cifar10-AR,concept,198.4,11.4,17.4,87.3,3.7,23.6
9,cifar10-AR,target,111.8,3.6,31.1,88.2,3.4,25.9
